# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster KMeans"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
36,2022-09-02 12:00:00,27523.885172,21,57,4,12,Nublado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,45,5,13,Nublado,Nublado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,37,6,14,Nublado,Nublado,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,33,5,15,Nublado,Nublado,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,34,4,16,Nublado,Nublado,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,36,2,17,Nublado,Nublado,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,39,1,18,Nublado,Nublado,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,44,1,19,Nublado,Nublado,22733.515002,18282.505369
60,2022-09-03 12:00:00,17723.695569,21,58,4,12,Nublado,Soleado,11246.659307,27523.885172
61,2022-09-03 13:00:00,20400.000000,23,48,5,13,Nublado,Nublado,17723.695569,20596.278869


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,21,57,4,12,17036.043251,29196.986647
37,23,45,5,13,27523.885172,25478.471342
38,24,37,6,14,20596.278869,29057.585772
39,26,33,5,15,28500.000000,30000.000000
40,27,34,4,16,24647.568577,28062.328964
...,...,...,...,...,...,...
18281,26,31,4,16,25562.000000,25385.000000
18282,26,32,2,17,25386.000000,22664.000000
18283,25,33,1,18,22872.000000,15736.000000
18284,23,38,0,19,15825.000000,1407.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
36,27523.885172
37,20596.278869
38,28500.000000
39,24647.568577
40,25500.000000
...,...
18281,25386.000000
18282,22872.000000
18283,15825.000000
18284,1450.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4641, y_train: 4641
X_val: 994, y_val: 994
X_test: 995, y_test: 995


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.43333333 0.7761194  0.28571429 0.4        0.56786811 0.97323289]
 [0.5        0.59701493 0.35714286 0.46666667 0.91746284 0.84928238]
 [0.53333333 0.47761194 0.42857143 0.53333333 0.68654263 0.96858619]
 ...
 [0.53333333 0.08955224 0.28571429 0.4        0.89256667 0.71003333]
 [0.6        0.04477612 0.28571429 0.46666667 0.89183333 0.6942    ]
 [0.66666667 0.02985075 0.28571429 0.53333333 0.884      0.67946667]]
(4641, 6)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.433333,0.776119,0.285714,0.400000,0.567868,0.973233
37,0.500000,0.597015,0.357143,0.466667,0.917463,0.849282
38,0.533333,0.477612,0.428571,0.533333,0.686543,0.968586
39,0.600000,0.417910,0.357143,0.600000,0.950000,1.000000
40,0.633333,0.432836,0.285714,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
13283,0.266667,0.253731,0.142857,0.266667,0.261667,0.703733
13284,0.400000,0.149254,0.214286,0.333333,0.847233,0.716300
13285,0.533333,0.089552,0.285714,0.400000,0.892567,0.710033
13286,0.600000,0.044776,0.285714,0.466667,0.891833,0.694200


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.7        0.02985075 0.21428571 0.6        0.88153333 0.68466667]
 [0.76666667 0.02985075 0.14285714 0.66666667 0.89283333 0.6928    ]
 [0.8        0.01492537 0.07142857 0.73333333 0.89383333 0.66686667]
 ...
 [0.73333333 0.41791045 0.85714286 0.4        0.94243333 0.95253333]
 [0.8        0.29850746 1.         0.46666667 0.94176667 0.95486667]
 [0.83333333 0.2238806  0.85714286 0.53333333 0.93916667 0.97816667]]
(994, 6)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
13288,0.700000,0.029851,0.214286,0.600000,0.881533,0.684667
13289,0.766667,0.029851,0.142857,0.666667,0.892833,0.692800
13290,0.800000,0.014925,0.071429,0.733333,0.893833,0.666867
13291,0.766667,0.014925,0.000000,0.800000,0.853433,0.491367
13292,0.700000,0.029851,0.000000,0.866667,0.514800,0.095967
...,...,...,...,...,...,...
15323,0.533333,0.805970,0.357143,0.266667,0.846533,0.940067
15324,0.633333,0.582090,0.642857,0.333333,0.911033,0.968733
15325,0.733333,0.417910,0.857143,0.400000,0.942433,0.952533
15326,0.800000,0.298507,1.000000,0.466667,0.941767,0.954867


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.9        0.1641791  0.64285714 0.6        0.93993333 0.95946667]
 [0.96666667 0.11940299 0.35714286 0.66666667 0.9288     0.94446667]
 [0.93333333 0.11940299 0.21428571 0.73333333 0.8208     0.89293333]
 ...
 [0.56666667 0.41791045 0.07142857 0.8        0.7624     0.52453333]
 [0.5        0.49253731 0.         0.86666667 0.5275     0.0469    ]
 [0.46666667 0.59701493 0.         0.93333333 0.04833333 0.        ]]
(995, 6)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15328,0.900000,0.164179,0.642857,0.600000,0.939933,0.959467
15329,0.966667,0.119403,0.357143,0.666667,0.928800,0.944467
15330,0.933333,0.119403,0.214286,0.733333,0.820800,0.892933
15331,0.900000,0.134328,0.142857,0.800000,0.788567,0.768000
15332,0.833333,0.179104,0.071429,0.866667,0.689433,0.349533
...,...,...,...,...,...,...
18281,0.600000,0.388060,0.285714,0.666667,0.852067,0.846167
18282,0.600000,0.402985,0.142857,0.733333,0.846200,0.755467
18283,0.566667,0.417910,0.071429,0.800000,0.762400,0.524533
18284,0.500000,0.492537,0.000000,0.866667,0.527500,0.046900


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.41935484 0.76056338 0.28571429 0.4        0.56786811 0.97323289]
 [0.48387097 0.5915493  0.35714286 0.46666667 0.91746284 0.84928238]
 [0.51612903 0.47887324 0.42857143 0.53333333 0.68654263 0.96858619]
 ...
 [0.5483871  0.42253521 0.07142857 0.8        0.7624     0.52453333]
 [0.48387097 0.49295775 0.         0.86666667 0.5275     0.0469    ]
 [0.4516129  0.5915493  0.         0.93333333 0.04833333 0.        ]]
(6630, 6)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.419355,0.760563,0.285714,0.400000,0.567868,0.973233
37,0.483871,0.591549,0.357143,0.466667,0.917463,0.849282
38,0.516129,0.478873,0.428571,0.533333,0.686543,0.968586
39,0.580645,0.422535,0.357143,0.600000,0.950000,1.000000
40,0.612903,0.436620,0.285714,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
18281,0.580645,0.394366,0.285714,0.666667,0.852067,0.846167
18282,0.580645,0.408451,0.142857,0.733333,0.846200,0.755467
18283,0.548387,0.422535,0.071429,0.800000,0.762400,0.524533
18284,0.483871,0.492958,0.000000,0.866667,0.527500,0.046900


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.89183333]
 [0.884     ]
 [0.88153333]]
(4641, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
13283,0.847233
13284,0.892567
13285,0.891833
13286,0.884000


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[8.92833333e-01]
 [8.93833333e-01]
 [8.53433333e-01]
 [5.14800000e-01]
 [5.18333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.03733333e-01]
 [6.26766667e-01]
 [7.09333333e-01]
 [7.83833333e-01]
 [6.76500000e-01]
 [6.85366667e-01]
 [6.87533333e-01]
 [6.60733333e-01]
 [4.83800000e-01]
 [9.24333333e-02]
 [0.00000000e+00]
 [6.76500000e-01]
 [6.89666667e-01]
 [7.04466667e-01]
 [6.53366667e-01]
 [4.85666667e-01]
 [6.37866667e-01]
 [6.10033333e-01]
 [5.91933333e-01]
 [5.99333333e-01]
 [6.08266667e-01]
 [6.53366667e-01]
 [4.23333333e-01]
 [9.31666667e-02]
 [8.27966667e-01]
 [7.89800000e-01]
 [8.75533333e-01]
 [8.66100000e-01]
 [8.54000000e-01]
 [7.82533333e-01]
 [7.58700000e-01]
 [5.93666667e-01]
 [1.34566667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.95366667e-01]
 [8.78733333e-01]
 [8.57166667e-01]
 [8.45600000e-01]
 [8.68233333e-01]
 [8.61400000e-01]
 [8.16733333e-01]
 [6.21266667e-01]
 [1.01333333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [9.48333333e-01]
 [9.95633333e-01]
 [9.944666

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
13288,0.892833
13289,0.893833
13290,0.853433
13291,0.514800
13292,0.051833
...,...
15323,0.911033
15324,0.942433
15325,0.941767
15326,0.939167


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.9288    ]
 [0.8208    ]
 [0.78856667]
 [0.68943333]
 [0.35676667]
 [0.0301    ]
 [0.        ]
 [0.7288    ]
 [0.77463333]
 [0.7569    ]
 [0.75133333]
 [0.8674    ]
 [0.77346667]
 [0.85126667]
 [0.78006667]
 [0.69566667]
 [0.35676667]
 [0.0335    ]
 [0.        ]
 [0.8673    ]
 [0.94856667]
 [0.97976667]
 [0.9742    ]
 [0.98153333]
 [0.94706667]
 [0.94543333]
 [0.9458    ]
 [0.7962    ]
 [0.73963333]
 [0.37316667]
 [0.02813333]
 [0.        ]
 [0.84413333]
 [0.91103333]
 [0.94243333]
 [0.94176667]
 [0.94386667]
 [0.9725    ]
 [0.9308    ]
 [0.7296    ]
 [0.80933333]
 [0.78803333]
 [0.40506667]
 [0.03346667]
 [0.        ]
 [0.8243    ]
 [0.91103333]
 [0.943     ]
 [0.94176667]
 [0.94886667]
 [0.95573333]
 [0.95593333]
 [0.9241    ]
 [0.86676667]
 [0.76606667]
 [0.42396667]
 [0.0338    ]
 [0.        ]
 [0.8237    ]
 [0.91103333]
 [0.9482    ]
 [0.94176667]
 [0.9482    ]
 [0.96123333]
 [0.95666667]
 [0.91926667]
 [0.86676667]
 [0.6952    ]
 [0.37006667]
 [0.03346667]
 [0.        ]
 [0.91

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15328,0.928800
15329,0.820800
15330,0.788567
15331,0.689433
15332,0.356767
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.5275    ]
 [0.04833333]
 [0.        ]]
(6630, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.315482 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 4641, number of used features: 6
[LightGBM] [Info] Start training from score 0.608742
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
15328,27737.854623
15329,27486.133289
15330,24127.781396
15331,20469.172397
15332,10001.014624
...,...
18281,27249.220044
18282,22924.525455
18283,14677.255501
18284,2147.631109


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
15328,27864.0
15329,24624.0
15330,23657.0
15331,20683.0
15332,10703.0
...,...
18281,25386.0
18282,22872.0
18283,15825.0
18284,1450.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
15328,27864.0,27737.854623
15329,24624.0,27486.133289
15330,23657.0,24127.781396
15331,20683.0,20469.172397
15332,10703.0,10001.014624
...,...,...
18281,25386.0,27249.220044
18282,22872.0,22924.525455
18283,15825.0,14677.255501
18284,1450.0,2147.631109


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1509.1542
RMSE: 2532.1425
R²: 0.9059


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
15328,27864.0,27737.854623,27676.1575
15329,24624.0,27486.133289,27436.08
15330,23657.0,24127.781396,24350.938017
15331,20683.0,20469.172397,20043.384431
15332,10703.0,10001.014624,10626.32
...,...,...,...
18281,25386.0,27249.220044,26034.882309
18282,22872.0,22924.525455,23229.408141
18283,15825.0,14677.255501,15735.242934
18284,1450.0,2147.631109,2056.066838


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4593, 48, 6), y_train: (4593, 1)
X_val: (946, 48, 6), y_val: (946, 1)
X_test: (947, 48, 6), y_test: (947, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 167s 4s/step - loss: 0.5130 - mean_absolute_error: 0.6209 - mean_absolute_percentage_error: 219284.8906 - root_mean_squared_error: 0.7162 - val_loss: 0.3864 - val_mean_absolute_error: 0.5393 - val_mean_absolute_percentage_error: 1300079.7500 - val_root_mean_squared_error: 0.6216
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - loss: 0.4971 - mean_absolute_error: 0.6105 - mean_absolute_percentage_error: 1390103.2500 - root_mean_squared_error: 0.7051 - val_loss: 0.3710 - val_mean_absolute_error: 0.5288 - val_mean_absolute_percentage_error: 3022328.2500 - val_root_mean_squared_error: 0.6091
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 35s 3s/step - loss: 0.4870 - mean_absolute_error: 0.6063 - mean_absolute_percentage_error: 3020765.2500 - root_mean_squared_error: 0.6978 - val_loss: 0.3505 - val_mean_absolute_error: 0.5150 - val_mean_absolute_percentage_error: 5394609.0000 - val_root_mean_squared_error: 0.5920
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - lo

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 345ms/step


array([[1.0051659 ],
       [0.9844358 ],
       [0.9281856 ],
       [0.82589674],
       [0.6822431 ],
       [0.5137037 ],
       [0.413108  ],
       [0.424401  ],
       [0.46983272],
       [0.589653  ],
       [0.71471566],
       [0.90906227],
       [0.99732184],
       [1.0190831 ],
       [1.0057576 ],
       [0.9430485 ],
       [0.8490745 ],
       [0.71833706],
       [0.5719695 ],
       [0.46737096],
       [0.42903793],
       [0.44474506],
       [0.57215726],
       [0.7105497 ],
       [0.9269676 ],
       [1.0104309 ],
       [1.0308535 ],
       [1.0132079 ],
       [0.9524594 ],
       [0.831304  ],
       [0.6687016 ],
       [0.4870611 ],
       [0.31443477],
       [0.30039173],
       [0.45712364],
       [0.67290497],
       [0.9284841 ],
       [1.0110314 ],
       [1.0216775 ],
       [1.0044352 ],
       [0.95721793],
       [0.8643916 ],
       [0.68307805],
       [0.43146735],
       [0.28537363],
       [0.30590123],
       [0.45054734],
       [0.648

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15328,27864.0,27737.854623,27676.1575,NaN
15329,24624.0,27486.133289,27436.08,NaN
15330,23657.0,24127.781396,24350.938017,NaN
15331,20683.0,20469.172397,20043.384431,NaN
15332,10703.0,10001.014624,10626.32,NaN
...,...,...,...,...
18281,25386.0,27249.220044,26034.882309,28669.376953
18282,22872.0,22924.525455,23229.408141,28550.570312
18283,15825.0,14677.255501,15735.242934,22407.324219
18284,1450.0,2147.631109,2056.066838,16069.017578


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50


8/8 ━━━━━━━━━━━━━━━━━━━━ 265s 9s/step - loss: 0.4692 - mean_absolute_error: 0.5923 - mean_absolute_percentage_error: 4315734.5000 - root_mean_squared_error: 0.6844 - val_loss: 0.1393 - val_mean_absolute_error: 0.3402 - val_mean_absolute_percentage_error: 41023272.0000 - val_root_mean_squared_error: 0.3732
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 44s 5s/step - loss: 0.1742 - mean_absolute_error: 0.3572 - mean_absolute_percentage_error: 55173304.0000 - root_mean_squared_error: 0.4171 - val_loss: 0.1280 - val_mean_absolute_error: 0.2622 - val_mean_absolute_percentage_error: 85287856.0000 - val_root_mean_squared_error: 0.3578
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 57s 7s/step - loss: 0.1403 - mean_absolute_error: 0.3244 - mean_absolute_percentage_error: 64999780.0000 - root_mean_squared_error: 0.3746 - val_loss: 0.1019 - val_mean_absolute_error: 0.2766 - val_mean_absolute_percentage_error: 58134472.0000 - val_root_mean_squared_error: 0.3192
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 99s 8s/step - loss: 0

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

30/30 ━━━━━━━━━━━━━━━━━━━━ 38s 556ms/step


array([[ 0.9231799 ],
       [ 0.93275493],
       [ 0.9066461 ],
       [ 0.9251513 ],
       [ 0.90455097],
       [ 0.9325918 ],
       [ 0.7116323 ],
       [ 0.4825462 ],
       [ 0.06643067],
       [ 0.00985141],
       [ 0.78841317],
       [ 0.80829096],
       [ 0.8828202 ],
       [ 0.9338947 ],
       [ 0.9249007 ],
       [ 0.896619  ],
       [ 0.9282765 ],
       [ 0.915404  ],
       [ 0.92563146],
       [ 0.7448062 ],
       [ 0.42824343],
       [ 0.0573285 ],
       [ 0.01263061],
       [ 0.7984353 ],
       [ 0.97259206],
       [ 0.94351447],
       [ 0.93121165],
       [ 0.9197299 ],
       [ 0.9079058 ],
       [ 0.9026924 ],
       [ 0.8444239 ],
       [ 0.6743317 ],
       [ 0.29396802],
       [ 0.03359183],
       [ 0.11938273],
       [ 0.82338786],
       [ 0.8983918 ],
       [ 0.9402831 ],
       [ 0.9267015 ],
       [ 0.8943697 ],
       [ 0.93465585],
       [ 0.92359567],
       [ 0.86624485],
       [ 0.5976077 ],
       [ 0.19347075],
       [ 0

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15328,27864.0,27737.854623,27676.1575,NaN
15329,24624.0,27486.133289,27436.08,NaN
15330,23657.0,24127.781396,24350.938017,NaN
15331,20683.0,20469.172397,20043.384431,NaN
15332,10703.0,10001.014624,10626.32,NaN
...,...,...,...,...
18281,25386.0,27249.220044,26034.882309,23223.203125
18282,22872.0,22924.525455,23229.408141,22499.527344
18283,15825.0,14677.255501,15735.242934,15411.343750
18284,1450.0,2147.631109,2056.066838,2882.268066


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,601 (1.55 MB)

 Trainable params: 405,345 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 673s 548ms/step - loss: 0.1583 - mae: 0.4603 - val_loss: 0.0535 - val_mae: 0.2761
Epoch 2/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 275s 468ms/step - loss: 0.1040 - mae: 0.3641 - val_loss: 0.0379 - val_mae: 0.1964
Epoch 3/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 268s 460ms/step - loss: 0.0789 - mae: 0.3144 - val_loss: 0.0165 - val_mae: 0.1326
Epoch 4/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 271s 463ms/step - loss: 0.0722 - mae: 0.2945 - val_loss: 0.0192 - val_mae: 0.1354
Epoch 5/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 313s 434ms/step - loss: 0.0594 - mae: 0.2643 - val_loss: 0.0142 - val_mae: 0.1203
Epoch 6/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 245s 391ms/step - loss: 0.0560 - mae: 0.2564 - val_loss: 0.0143 - val_mae: 0.1130
Epoch 7/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 291s 434ms/step - loss: 0.0489 - mae: 0.2378 - val_loss: 0.0172 - val_mae: 0.1178
Epoch 8/100
575/575 ━━━━━━━━━━━━━━━━━━━━ 264s 423ms/step - loss: 0.0458 - mae: 0.2271 - val_loss: 0.0129 - val_mae: 0.1122
Epoch 9/100
575/

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

30/30 ━━━━━━━━━━━━━━━━━━━━ 28s 689ms/step


array([[0.90593344],
       [0.89468503],
       [0.8679791 ],
       [0.8659721 ],
       [0.8694522 ],
       [0.8163233 ],
       [0.66185194],
       [0.3333025 ],
       [0.        ],
       [0.        ],
       [0.6921209 ],
       [0.8798448 ],
       [0.86563236],
       [0.8994745 ],
       [0.8923542 ],
       [0.8746653 ],
       [0.872284  ],
       [0.88262314],
       [0.82266086],
       [0.7106226 ],
       [0.4136085 ],
       [0.        ],
       [0.        ],
       [0.75302255],
       [0.9105719 ],
       [0.88341695],
       [0.8959858 ],
       [0.8710844 ],
       [0.8619146 ],
       [0.75655013],
       [0.7633082 ],
       [0.63623786],
       [0.27426973],
       [0.        ],
       [0.        ],
       [0.7483559 ],
       [0.86896545],
       [0.8295423 ],
       [0.8873877 ],
       [0.8794565 ],
       [0.8727062 ],
       [0.90841347],
       [0.81934977],
       [0.62559754],
       [0.29584363],
       [0.        ],
       [0.        ],
       [0.771

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
15328,27864.0,27737.854623,27676.1575,NaN,NaN
15329,24624.0,27486.133289,27436.08,NaN,NaN
15330,23657.0,24127.781396,24350.938017,NaN,NaN
15331,20683.0,20469.172397,20043.384431,NaN,NaN
15332,10703.0,10001.014624,10626.32,NaN,NaN
...,...,...,...,...,...
18281,25386.0,27249.220044,26034.882309,23223.203125,22693.320312
18282,22872.0,22924.525455,23229.408141,22499.527344,21046.179688
18283,15825.0,14677.255501,15735.242934,15411.343750,12242.646484
18284,1450.0,2147.631109,2056.066838,2882.268066,0.000000


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

15421    28253.0
15422    28466.0
15423    28672.0
15424    28678.0
15425    27723.0
          ...   
18281    25386.0
18282    22872.0
18283    15825.0
18284     1450.0
18285        0.0
Name: Generación, Length: 947, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      1,600 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 473,281 (1.81 MB)

 Trainable params: 473,281 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 174s 151ms/step - loss: 1.1302 - mae: 0.3101 - val_loss: 0.5877 - val_mae: 0.3451
Epoch 2/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 32s 97ms/step - loss: 0.4581 - mae: 0.2543 - val_loss: 0.2157 - val_mae: 0.1701
Epoch 3/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - loss: 0.1826 - mae: 0.1560 - val_loss: 0.0988 - val_mae: 0.1214
Epoch 4/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 42s 87ms/step - loss: 0.0949 - mae: 0.1307 - val_loss: 0.0654 - val_mae: 0.1318
Epoch 5/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 43s 87ms/step - loss: 0.0621 - mae: 0.1254 - val_loss: 0.0650 - val_mae: 0.1642
Epoch 6/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 46s 94ms/step - loss: 0.0491 - mae: 0.1243 - val_loss: 0.0348 - val_mae: 0.1087
Epoch 7/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 43s 99ms/step - loss: 0.0403 - mae: 0.1186 - val_loss: 0.0283 - val_mae: 0.0964
Epoch 8/50
288/288 ━━━━━━━━━━━━━━━━━━━━ 41s 93ms/step - loss: 0.0342 - mae: 0.1131 - val_loss: 0.0375 - val_mae: 0.1335
Epoch 9/50
288/288 ━━━━━━━━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

30/30 ━━━━━━━━━━━━━━━━━━━━ 25s 219ms/step


array([[0.944848  ],
       [0.9413351 ],
       [0.89030194],
       [0.9196884 ],
       [0.88631845],
       [0.83406425],
       [0.71732914],
       [0.4793254 ],
       [0.00896139],
       [0.00492401],
       [0.44046012],
       [0.84171927],
       [0.88459235],
       [0.9415931 ],
       [0.9180594 ],
       [0.90378505],
       [0.9089012 ],
       [0.8947825 ],
       [0.8709169 ],
       [0.75766456],
       [0.44970965],
       [0.01809145],
       [0.01185508],
       [0.62786245],
       [0.93421346],
       [0.9429063 ],
       [0.9565627 ],
       [0.9267876 ],
       [0.90782785],
       [0.8534126 ],
       [0.8263086 ],
       [0.7197625 ],
       [0.30606046],
       [0.01599721],
       [0.02010973],
       [0.66034687],
       [0.89905584],
       [0.8755655 ],
       [0.9363746 ],
       [0.9323505 ],
       [0.895312  ],
       [0.9152453 ],
       [0.8792418 ],
       [0.66820395],
       [0.2719837 ],
       [0.01914129],
       [0.01500765],
       [0.628

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
15328,27864.0,27737.854623,27676.1575,NaN,NaN,NaN
15329,24624.0,27486.133289,27436.08,NaN,NaN,NaN
15330,23657.0,24127.781396,24350.938017,NaN,NaN,NaN
15331,20683.0,20469.172397,20043.384431,NaN,NaN,NaN
15332,10703.0,10001.014624,10626.32,NaN,NaN,NaN
...,...,...,...,...,...,...
18281,25386.0,27249.220044,26034.882309,23223.203125,22693.320312,24257.355469
18282,22872.0,22924.525455,23229.408141,22499.527344,21046.179688,23507.683594
18283,15825.0,14677.255501,15735.242934,15411.343750,12242.646484,16994.105469
18284,1450.0,2147.631109,2056.066838,2882.268066,0.000000,2008.016968


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1509.1542
RMSE: 2532.1425
R²: 0.9059
Random Forest
MAE: 1666.8296
RMSE: 3025.3681
R²: 0.8657
CTNET
MAE: 3449.8432
RMSE: 4889.9729
R²: 0.6403
Forecast
MAE: 3800.3562
RMSE: 5225.2845
R²: 0.5893
Photovoltaic
MAE: 3210.1219
RMSE: 4913.9671
R²: 0.6368


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

144/144 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

144/144 ━━━━━━━━━━━━━━━━━━━━ 15s 68ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

144/144 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
36,27523.885172,21098.213009,23664.907604,NaN,NaN,NaN
37,20596.278869,26744.462625,23380.516505,NaN,NaN,NaN
38,28500.000000,22151.143748,25756.058073,NaN,NaN,NaN
39,24647.568577,27274.650131,25507.277645,NaN,NaN,NaN
40,25500.000000,24396.094841,25178.124908,NaN,NaN,NaN
...,...,...,...,...,...,...
13283,25417.000000,21338.219266,23812.422852,25327.320312,22362.228516,16248.473633
13284,26777.000000,26322.373558,26513.410000,22346.621094,22139.007812,18893.138672
13285,26755.000000,26662.111071,26720.286252,24434.644531,26526.132812,23050.728516
13286,26520.000000,26047.924317,26574.692071,25905.248047,27783.910156,25191.732422


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1023.3294
RMSE: 1845.5986
R²: 0.9693
Random Forest
MAE: 381.6334
RMSE: 739.3641
R²: 0.9951
CTNET
MAE: 2548.7851
RMSE: 3779.4295
R²: 0.8719
Forecast
MAE: 2604.5511
RMSE: 3853.4461
R²: 0.8668
Photovoltaic
MAE: 2500.8966
RMSE: 3935.4282
R²: 0.8611


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.9_Predicciones_Conjunto_nublado KMeans CTNET.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_9_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_9_RandomForest_model.pkl")


['4_9_RandomForest_model.pkl']

In [87]:
CTNET.save("4_9_CTNET_model.keras")
Forecast_model.save("4_9_Forecast_model.keras")
Photo_model.save("4_9_Photo_model.keras")